In [58]:
# 코드 실행에 필요한 라이브러리를 미리 import.
import csv
import os
import platform
import torch

import imageio.v2 as imageio
import numpy as np

from PIL import Image
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

In [59]:
if platform.system() == 'Windows':
    HOME = 'C:/Users/first/'
else:
    HOME = '/home/ksy/'
PATH_PREFIX = f'{HOME}Develop/Kut-Deep-Learning-250204/'

# a_2d_image_data

In [60]:
# 이미지 읽기.
img_arr = imageio.imread(os.path.join(PATH_PREFIX, "_00_data", "a_image-dog", "bobby.jpg"))
print(type(img_arr))        # 이미지는 n차원 배열로 표현됨.
print(img_arr.shape)        # (rows, cols, channels)
print(img_arr.dtype)        # 각 필셀은 [0, 256) 범위의 값을 가짐 -> unit8

img = torch.from_numpy(img_arr)     # 넘파이 배열로부터 토치에서 사용할 수 있는 형태로 변화.
out = img.permute(2, 0, 1)          # (channels, rows, cols)
print(out.shape)

<class 'numpy.ndarray'>
(720, 1280, 3)
uint8
torch.Size([3, 720, 1280])


In [61]:
# 이미지 파일 이름 불러오기.
data_dir = os.path.join(PATH_PREFIX, "_00_data", "b_image-cats")
filenames = [
    name for name in os.listdir(data_dir) if os.path.splitext(name)[-1] == '.png'
]
print(filenames)

# 이미지 순회하기.
for i, filename in enumerate(filenames):
    image = Image.open(os.path.join(data_dir, filename))
    # image.show()  # 이미지 뷰어에서 열기.
    img_arr = imageio.imread(os.path.join(data_dir, filename))
    print(img_arr.shape)      # (rows, cols, channels)
    print(img_arr.dtype)      # uint8

# 넘파이 배열을 토치 텐서로 변환하기 위한 준비.
# 각 이미지는 하나의 batch 단위.
batch_size = 3
# (255, 255, 3)이 들어갈 공간이 세 개 필요하므로 (3, 256, 256, 3)
# 그런데 토치에서는 channels이 먼저 오므로 (3, 3, 256, 256)
batch = torch.zeros(batch_size, 3, 256, 256, dtype=torch.uint8)

# 이미지 순회하며,
for i, filename in enumerate(filenames):
    img_arr = imageio.imread(os.path.join(data_dir, filename))  # 이미지 읽고,
    img_t = torch.from_numpy(img_arr)                           # 넘파이 -> 토치 변환하고,
    img_t = img_t.permute(2, 0, 1)                              # channels가 먼저 오도록 하고,
    batch[i] = img_t                                            # batch로 복사.

# (3, 3, 256, 256)
print(batch.shape)

['cat1.png', 'cat2.png', 'cat3.png']
(256, 256, 3)
uint8
(256, 256, 3)
uint8
(256, 256, 3)
uint8
torch.Size([3, 3, 256, 256])


In [62]:
# 정규화.
batch = batch.float()   # uint8 -> float32
batch /= 255.0          # [0, 256) -> [0, 1)
print(batch.dtype)      # float32
print(batch.shape)      # (3, 3, 256, 256)

# RGB 포맷 파싱. (색상 채널이 몇 개인가)
n_channels = batch.shape[1]

# 채널 별로 순회.
for c in range(n_channels):
    mean = torch.mean(batch[:, c])      # 현재 색상의 전체 픽셀 평균.
    std = torch.std(batch[:, c])        # 현재 색상의 전체 픽셀 표준편차.
    print(mean, std)
    batch[:, c] = (batch[:, c] - mean) / std    # 평균: 0, 표준편차: 단위 표준편차.

torch.float32
torch.Size([3, 3, 256, 256])
tensor(0.5799) tensor(0.2212)
tensor(0.4493) tensor(0.2068)
tensor(0.3554) tensor(0.1931)


# b_tabular_wine_data_to_tensors

In [63]:
wine_path = os.path.join(PATH_PREFIX, "_00_data", "d_tabular-wine", "winequality-white.csv")
# CSV 형식 불러오기. 단, 구분자는 `;`.
# 첫 행 무시. 컬럼명.
wineq_numpy = np.loadtxt(wine_path, dtype=np.float32, delimiter=";", skiprows=1)
print(wineq_numpy.dtype)
print(wineq_numpy.shape)
print(wineq_numpy)
print()

# 컬럼명 가져오기.
# 파일의 내용을 불러오는 제너레이터로부터 첫 원소만 가져옴.
col_list = next(csv.reader(open(wine_path), delimiter=';'))
print(col_list)
print()

float32
(4898, 12)
[[ 7.    0.27  0.36 ...  0.45  8.8   6.  ]
 [ 6.3   0.3   0.34 ...  0.49  9.5   6.  ]
 [ 8.1   0.28  0.4  ...  0.44 10.1   6.  ]
 ...
 [ 6.5   0.24  0.19 ...  0.46  9.4   6.  ]
 [ 5.5   0.29  0.3  ...  0.38 12.8   7.  ]
 [ 6.    0.21  0.38 ...  0.32 11.8   6.  ]]

['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality']



In [64]:
# 넘파이 -> 토치 형식 변환.
wineq = torch.from_numpy(wineq_numpy)
print(wineq.dtype)
print(wineq.shape)
print()

# 파이썬에서 `:-1`은 [begin, end - 1)을 의미.
# 즉, 마지막을 제외한 모든 열.
data = wineq[:, :-1]  # Selects all rows and all columns except the last
print(data.dtype)
print(data.shape)
print(data)
print()

# 파이썬에서 `-1`은 end - 1을 의미.
# 즉, 마지막을 열.
target = wineq[:, -1]  # Selects all rows and the last column
print(target.dtype)
print(target.shape)
print(target)
print()

# 자료형 변환: float32 -> int64
target = target.to(torch.int64)  # treat labels as an integer
print(target.dtype)
print(target.shape)
print(target)
print()

torch.float32
torch.Size([4898, 12])

torch.float32
torch.Size([4898, 11])
tensor([[ 7.0000,  0.2700,  ...,  0.4500,  8.8000],
        [ 6.3000,  0.3000,  ...,  0.4900,  9.5000],
        ...,
        [ 5.5000,  0.2900,  ...,  0.3800, 12.8000],
        [ 6.0000,  0.2100,  ...,  0.3200, 11.8000]])

torch.float32
torch.Size([4898])
tensor([6., 6.,  ..., 7., 6.])

torch.int64
torch.Size([4898])
tensor([6, 6,  ..., 7, 6])



In [65]:
# 10 * 10크기의 단위 행렬 생성.
# [1, 0, 0, ..., 0]
# [0, 1, 0, ..., 0]
# [0, 0, 1, ..., 0]
# ...
# [0, 0, 0, ..., 1]
eye_matrix = torch.eye(10)
# One-Hot Encoding: 원하는 열에 1, 원하는 열을 제외한 나머지 열에 0을 부여하는 방식.
# e.g.
#     데이터가 [a, b, c, d]일 때,
#     onehot을 [0, 1, 0, 0]으로 설정하면,
#     원하는 데이터는 `b`임을 알 수 있음.
# We use the 'target' tensor as indices to extract the corresponding rows from the identity matrix
# It can generate the one-hot vectors for each element in the 'target' tensor
# target이 n이면,[6, 6, 6,  ..., 6, 7, 6]
# [0, 0, 0, ..., 1, 0, 0, ..., 0]으로 변환.
#  ^^^^^^^^^^^^^
#  0이 n - 1개
onehot_target = eye_matrix[target]

# 데이터의 개수가 4989인 데이터에서 각각 길이가 10인 onehot 벡터를 추출하므로,
# 크기는 (4898, 10)이 된다.
# target: [6, 6, 6,  ..., 6, 7, 6]
# => [
#        [0, 0, 0, 0, 0, 0, 1, 0, ...],
#        [0, 0, 0, 0, 0, 0, 1, 0, ...],
#        [0, 0, 0, 0, 0, 0, 1, 0, ...],
#    ...
#        [0, 0, 0, 0, 0, 0, 1, 0, ...],
#        [0, 0, 0, 0, 0, 0, 0, 1, ...],
#        [0, 0, 0, 0, 0, 0, 1, 0, ...]
#    ]
print(onehot_target.shape)  # >>> torch.Size([4898, 10])
print(onehot_target[0])
print(onehot_target[1])
print(onehot_target[-2])
print(onehot_target)

torch.Size([4898, 10])
tensor([0., 0., 0., 0., 0., 0., 1., 0., 0., 0.])
tensor([0., 0., 0., 0., 0., 0., 1., 0., 0., 0.])
tensor([0., 0., 0., 0., 0., 0., 0., 1., 0., 0.])
tensor([[0., 0.,  ..., 0., 0.],
        [0., 0.,  ..., 0., 0.],
        ...,
        [0., 0.,  ..., 0., 0.],
        [0., 0.,  ..., 0., 0.]])


In [66]:
data_mean = torch.mean(data, dim=0)     # 각 데이터의 평균과,
data_var = torch.var(data, dim=0)       # 분산을 구해서,
data = (data - data_mean) / torch.sqrt(data_var)    # 데이터 표준화.
print(data)

tensor([[ 1.7208e-01, -8.1761e-02,  ..., -3.4915e-01, -1.3930e+00],
        [-6.5743e-01,  2.1587e-01,  ...,  1.3422e-03, -8.2419e-01],
        ...,
        [-1.6054e+00,  1.1666e-01,  ..., -9.6251e-01,  1.8574e+00],
        [-1.0129e+00, -6.7703e-01,  ..., -1.4882e+00,  1.0448e+00]])


In [67]:
# 주어진 데이터를 0.8 : 0.2로 나누어 학습 셋과 테스트 셋으로 나눔.
X_train, X_test, y_train, y_test = train_test_split(data, onehot_target, test_size=0.2)

print(X_train.shape)
print(y_train.shape)

print(X_test.shape)
print(y_test.shape)

torch.Size([3918, 11])
torch.Size([3918, 10])
torch.Size([980, 11])
torch.Size([980, 10])


In [68]:
# 위의 과정을 하나의 함수로 분리.
def get_wine_data():
    # 파일 불러오기.
    wine_path = os.path.join(PATH_PREFIX, "_00_data", "d_tabular-wine", "winequality-white.csv")
    wineq_numpy = np.loadtxt(wine_path, dtype=np.float32, delimiter=";", skiprows=1)

    # 토치 형식으로 변환하기.
    wineq = torch.from_numpy(wineq_numpy)

    # 입력과 출력 분리하기.
    data = wineq[:, :-1]  # Selects all rows and all columns except the last
    target = wineq[:, -1].to(torch.int64)  # treat labels as an integer

    # onehot 변환.
    eye_matrix = torch.eye(10)
    onehot_target = eye_matrix[target]

    # 데이터 표준화.
    data_mean = torch.mean(data, dim=0)
    data_var = torch.var(data, dim=0)
    data = (data - data_mean) / torch.sqrt(data_var)

    # 학습 셋과 테스트 셋 선별.
    X_train, X_valid, y_train, y_valid = train_test_split(data, onehot_target, test_size=0.2)

    return X_train, X_valid, y_train, y_valid

print(get_wine_data())

(tensor([[ 1.7208e-01,  1.7450e-02,  ...,  1.4033e+00,  4.7597e-01],
        [-1.8343e-01, -1.3715e+00,  ...,  1.3422e-03,  3.1345e-01],
        ...,
        [ 4.0908e-01, -3.7940e-01,  ..., -6.1202e-01, -7.4293e-01],
        [ 5.3577e-02, -1.2723e+00,  ...,  7.8995e-01,  5.5723e-01]]), tensor([[-0.5389, -0.5778,  ..., -1.6635,  0.3947],
        [-1.2499,  0.6127,  ..., -0.3491, -0.4179],
        ...,
        [ 0.1721,  0.2159,  ...,  0.2642, -0.7429],
        [-0.4204,  0.1167,  ...,  0.1766, -0.8242]]), tensor([[0., 0.,  ..., 0., 0.],
        [0., 0.,  ..., 0., 0.],
        ...,
        [0., 0.,  ..., 0., 0.],
        [0., 0.,  ..., 0., 0.]]), tensor([[0., 0.,  ..., 0., 0.],
        [0., 0.,  ..., 0., 0.],
        ...,
        [0., 0.,  ..., 0., 0.],
        [0., 0.,  ..., 0., 0.]]))


# c_tabular_california_housing

In [69]:
housing = fetch_california_housing()
print(housing.keys())       # housing이 무슨 멤버 변수를 갖고 있는가?

print(type(housing.data))
print(housing.data.dtype)
print(housing.data.shape)
print(housing.feature_names)    # 데이터에 어떤 칼럼이 있는가?

print(housing.target.shape)
print(housing.target_names)     # (예측 시) 얻고자 하는 값.

dict_keys(['data', 'target', 'frame', 'target_names', 'feature_names', 'DESCR'])
<class 'numpy.ndarray'>
float64
(20640, 8)
['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
(20640,)
['MedHouseVal']


In [70]:
# data에 있는 값들 중 최소/최댓값.
print(housing.data.min(), housing.data.max())

# 데이터 표준화.
data_mean = np.mean(housing.data, axis=0)
data_var = np.var(housing.data, axis=0)
data = (housing.data - data_mean) / np.sqrt(data_var)
target = housing.target

print(data.min(), data.max())

# 표준화 후에는 평균이 0, 표준편차가 1이 됨.
print(data.mean(), data.std())

-124.35 35682.0
-2.3859923416733877 119.41910318829314
-7.864510076763318e-16 1.0


In [71]:
# 표준화한 데이터를 학습 셋과 테스트 셋으로 나누기.
X_train, X_test, y_train, y_test = train_test_split(data, target, test_size=0.2)

X_train = torch.from_numpy(X_train)
X_test = torch.from_numpy(X_test)
y_train = torch.from_numpy(y_train)
y_test = torch.from_numpy(y_test)

print(X_train.shape)
print(y_train.shape)

print(X_test.shape)
print(y_test.shape)

torch.Size([16512, 8])
torch.Size([16512])
torch.Size([4128, 8])
torch.Size([4128])


# d_bikes_sharing_data

In [72]:
# 출력 형식 설정.
# edgeitmes: 원소가 많이 일부만 출력할 때 양 끝에 올 원소의 개수.
# threshold: 전체 원소아 아닌 일부 원소만 출력할 원소 개수의 경계.
# linewidth: 출력 시 각 줄의 최대 길이.
torch.set_printoptions(edgeitems=2, threshold=50, linewidth=75)

# 데이터 경로 설정.
bikes_path = os.path.join(PATH_PREFIX, "_00_data", "e_time-series-bike-sharing-dataset", "hour-fixed.csv")

# 데이터 불러오기.
# CSV 형식으로 구분자는 `;`.
bikes_numpy = np.loadtxt(
    fname=bikes_path, dtype=np.float32, delimiter=",", skiprows=1,
    converters={
        # 날짜의 일을 추출해 float32로 변환.
        1: lambda x: float(x[8:10])  # 2011-01-07 --> 07 --> 7.0
    }
)
bikes = torch.from_numpy(bikes_numpy).to(torch.float)   # 넘파이 배열에서 토치 배열로 변환.
print(bikes.shape)

# 원본 데이터는 유지하되 shape만 바꾸어 본다.
# `-1`은 다른 dim의 크기를 통해 추론한다.
# (17520, 71) -> (?, 24, 17) -> (730, 24, 17)
daily_bikes = bikes.view(-1, 24, bikes.shape[1])
print(daily_bikes.shape)  # >>> torch.Size([730, 24, 17])

# 데이터와 타깃 분리.
daily_bikes_data = daily_bikes[:, :, :-1]
daily_bikes_target = daily_bikes[:, :, -1].unsqueeze(dim=-1)    # 타깃이 스칼라이므로 이를 unsqueeze하여 텐서로 변환.

print(daily_bikes_data.shape)
print(daily_bikes_target.shape)

torch.Size([17520, 17])
torch.Size([730, 24, 17])
torch.Size([730, 24, 16])
torch.Size([730, 24, 1])


In [73]:
# 첫 하루의 데이터 가져오기.
first_day_data = daily_bikes_data[0]
print(first_day_data.shape)

# Whether situation: 1: clear, 2:mist, 3: light rain/snow, 4: heavy rain/snow
# 날씨 종류를 onehot으로 나타내기 위한 준비 단계.
# *[:, 9]는 모든 행의 9번째 열 선택.
print(first_day_data[:, 9])
print(first_day_data[:, 9].shape, first_day_data[:, 9].dtype)
eye_matrix = torch.eye(4)
print(eye_matrix)

# 날씨 정보를 onehot으로 변환.
# 날시 종류가 1-based이므로 1을 빼줘야 한다.
weather_onehot = eye_matrix[first_day_data[:, 9].to(torch.int64) - 1]
print(weather_onehot.shape)
print(weather_onehot)

first_day_data_torch = torch.cat(tensors=(first_day_data, weather_onehot), dim=1)
print(first_day_data_torch.shape)
print(first_day_data_torch)

# first_day_data에서 날씨 정보를 추출해 날씨 정보의 onehot 벡터를 추가한다.

torch.Size([24, 16])
tensor([1., 1., 1., 1., 1., 2., 1., 1., 1., 1., 1., 1., 1., 2., 2., 2., 2.,
        2., 3., 3., 2., 2., 2., 2.])
torch.Size([24]) torch.float32
tensor([[1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 0., 0., 1.]])
torch.Size([24, 4])
tensor([[1., 0., 0., 0.],
        [1., 0., 0., 0.],
        ...,
        [0., 1., 0., 0.],
        [0., 1., 0., 0.]])
torch.Size([24, 20])
tensor([[ 1.,  1.,  ...,  0.,  0.],
        [ 2.,  1.,  ...,  0.,  0.],
        ...,
        [23.,  1.,  ...,  0.,  0.],
        [24.,  1.,  ...,  0.,  0.]])


In [74]:
day_data_torch_list = []

# 모든 날짜의 데이터에 대해,
for daily_idx in range(daily_bikes_data.shape[0]):  # range(730)
    # 그날의 데이터를 추출하고,
    day = daily_bikes_data[daily_idx]  # day.shape: [24, 16]
    # 날씨 종류의 onehot 벡터를 만들어서,
    weather_onehot = eye_matrix[day[:, 9].to(torch.int64) - 1]
    # 데이터의 맨 끝에 병합한다.
    day_data_torch = torch.cat(tensors=(day, weather_onehot), dim=1)  # day_data_torch.shape: [24, 20]
    # 마지막으로 모든 데이터 취합.
    day_data_torch_list.append(day_data_torch)

print(len(day_data_torch_list))
daily_bikes_data = torch.stack(day_data_torch_list, dim=0)  # 파이썬 리스트를 토치 텐서로 변환.
print(daily_bikes_data.shape)

730
torch.Size([730, 24, 20])


In [75]:
print(daily_bikes_data[:, :, :9].shape, daily_bikes_data[:, :, 10:].shape)
# 0번째 열인 `instant`열과 9번째 열인 `whethersit`열 제거.
# [1:9]와 [10:] 병합.
daily_bikes_data = torch.cat(
    [daily_bikes_data[:, :, 1:9], daily_bikes_data[:, :, 10:]], dim=2
) # Drop 'instant' and 'whethersit' columns
print(daily_bikes_data.shape)

# instant와 whethersit을 제거한 데이터에서 온도 정보는 8번째 열.
temperatures = daily_bikes_data[:, :, 8]
# 온도 정보 표준화.
daily_bikes_data[:, :, 8] = (daily_bikes_data[:, :, 8] - torch.mean(temperatures)) / torch.std(temperatures)

# 평균 0, 표준편차 1.
print(daily_bikes_data[:, :, 8].mean(), daily_bikes_data[:, :, 8].std())

torch.Size([730, 24, 9]) torch.Size([730, 24, 10])
torch.Size([730, 24, 18])
tensor(6.2707e-08) tensor(1.)
